to do : 

- fetch all the macro data

Construire un score macro pour chaque zones monétaires et ensuite tester si il fonctionne pas pour predire les variations sur fx sur les paires de la zone en question 


API à utiliser: 

- fred
- alpha vantage
- reddit API (facultatif / test )



Hypothèses : 





In [3]:
import os
import sys
import yfinance as yf
import pandas as pd
from dotenv import load_dotenv

# Ajout du chemin parent pour accéder à macro_validator si nécessaire
sys.path.append("..") 

from data_fetch import get_macro_data
from engine import aggregate_macro_scores
from macro_validator import MacroValidator

# Chargement des variables d'environnement (FRED_API_KEY)
load_dotenv()


True

In [4]:
print("--- [1] EXTRACTION ET CALCUL DU Z-SCORE MACRO ---")

# 1. Extraction Point-in-Time (First Release)
print("Téléchargement des données macroéconomiques depuis FRED...")
data_map = get_macro_data()

# 2. Calcul vectorisé de la moyenne dynamique et normalisation Z-Score
print("Calcul des Z-Scores croisés (Cross-Zone Comparability)...")
df_scores = aggregate_macro_scores(data_map)

# Affichage des 5 derniers jours
display(df_scores.tail())


--- [1] EXTRACTION ET CALCUL DU Z-SCORE MACRO ---
Téléchargement des données macroéconomiques depuis FRED...
Calcul des Z-Scores croisés (Cross-Zone Comparability)...


,USD,EUR,GBP,AUD,NZD
2026-03-21,-0.129601,-0.420252,0.570683,-0.265509,0.560173
2026-03-22,-0.146107,-0.423565,0.569916,-0.301353,0.557068
2026-03-23,-0.160469,-0.426433,0.569245,-0.332590,0.554311
2026-03-24,-0.172960,-0.428918,0.568659,-0.359811,0.551870
2026-03-25,-0.183822,-0.431071,0.568148,-0.383528,0.549715


In [5]:
print("--- [2] PREPARATION DES DONNEES DE MARCHE (TARGET) ---")

# Téléchargement du SPY comme proxy de l'économie US
start_date = df_scores.index.min().strftime('%Y-%m-%d')
end_date = df_scores.index.max().strftime('%Y-%m-%d')

print(f"Téléchargement du SPY de {start_date} à {end_date}...")
spy = yf.download("EURUSD=X", start=start_date, end=end_date, progress=False)

if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.droplevel(1)

# Calcul du rendement futur à 1 mois (21 jours de trading)
# Le shift(-21) ramène le rendement futur à la date t du signal
spy['Forward_Return_21d'] = spy['Close'].pct_change(42).shift(-42)

# Extraction et alignement du signal USD (Forward-filled pour matcher les jours de trading)
signal_usd = df_scores['USD'].reindex(spy.index).ffill()
forward_returns = spy['Forward_Return_21d']


--- [2] PREPARATION DES DONNEES DE MARCHE (TARGET) ---
Téléchargement du SPY de 2016-10-31 à 2026-03-25...


In [6]:
print("--- [3] VALIDATION STATISTIQUE DE L'ALPHA (MACRO VALIDATOR) ---")

# Instanciation du Validateur
validator = MacroValidator()

# 1. Coefficient d'Information (IC) Global
ic_results = validator.compute_ic(signal_usd, forward_returns)
print("\n[RESULTATS IC GLOBAUX]")
print(f"Information Coefficient (IC) : {ic_results['IC']:.4f}")
print(f"P-Value                      : {ic_results['P-Value']:.4f}")
print(f"Nombre d'observations (N)    : {ic_results['N']}")

# 2. Rolling IC (Stabilité temporelle)
print("\nGénération du Rolling IC...")
fig_rolling_ic = validator.plot_rolling_ic(signal_usd, forward_returns, window=252)
fig_rolling_ic.show()

# 3. Analyse Factorielle par Quantiles (Monotonie)
print("\nGénération du Quintile Spread...")
fig_quantiles = validator.plot_quantile_analysis(signal_usd, forward_returns, quantiles=5)
fig_quantiles.show()


--- [3] VALIDATION STATISTIQUE DE L'ALPHA (MACRO VALIDATOR) ---

[RESULTATS IC GLOBAUX]
Information Coefficient (IC) : 0.0555
P-Value                      : 0.0065
Nombre d'observations (N)    : 2402

Génération du Rolling IC...



Génération du Quintile Spread...
